# 1.3 Memory — Making Agents Remember

## What you will learn in this notebook

- Why agents are **stateless by default** and what that means
- How to add **short-term (in-memory) memory** using LangGraph's `InMemorySaver`
- What a **`thread_id`** is and why it is the key to multi-user memory isolation
- How memory changes the agent's message history on each call

---

### The statelessness problem

By default, every `.invoke()` call is a **brand new conversation**. The agent has no memory of what was said in previous calls — even if they happened one second ago.

```
Call 1:  "Hi, my name is Seán and my favourite colour is green."
         Agent: "Nice to meet you, Seán!"

Call 2:  "What's my favourite colour?"
         Agent: "I don't know — I don't have any information about your preferences."
                ↑ The agent has completely forgotten Call 1!
```

This is fine for one-shot tasks (summarise this doc, translate this sentence) but breaks any **conversational** application where context carries across turns.

### The solution: Checkpointers

A **checkpointer** is a persistence backend that LangGraph uses to save and reload conversation state between `.invoke()` calls. Every time the agent finishes a turn, it writes the updated message history to the checkpointer. On the next call, it reads it back.

```
WITH checkpointer:

Call 1 (thread_id="1"):  "My name is Seán, favourite colour is green"
   → saved to checkpointer under thread_id="1"

Call 2 (thread_id="1"):  "What's my favourite colour?"
   → loads history for thread_id="1"
   → agent sees BOTH messages from Call 1
   → "Your favourite colour is green, Seán!"
```

### What is a `thread_id`?

A **thread** is one isolated conversation. The `thread_id` is a string key that identifies it.

- Same `thread_id` → the agent remembers the history from previous calls in that thread
- Different `thread_id` → completely separate conversation, no shared memory

In a real app, `thread_id` would be a **user session ID** or **chat room ID** — one per user, so users don't see each other's conversations.

In [1]:
# ============================================================
# CELL 1: Load Environment Variables
# ============================================================

from dotenv import load_dotenv

load_dotenv()

True

---
## Section 1 — No Memory (Default Behaviour)

In [4]:
# ============================================================
# CELL 2: Create a Stateless Agent (No Checkpointer)
# ============================================================
# Without a checkpointer, every .invoke() call starts completely
# fresh. There is no persistence between calls.
# This is the default behaviour — no special setup needed.

from langchain.agents import create_agent

agent = create_agent(
    "google_genai:gemini-2.5-flash"
    # No checkpointer= → stateless, no memory between calls
)

In [7]:
# ============================================================
# CELL 3: Turn 1 — Introduce Yourself to the Agent
# ============================================================
# We tell the agent two facts: name and favourite colour.
# The agent will respond positively, acknowledging both.

from langchain.messages import HumanMessage

question = HumanMessage(content="Hello my name is Yogi Reddy and my favourite colour is Blue")

response = agent.invoke(
    {"messages": [question]}
    # No config= here — stateless agents don't use thread_id
)

In [8]:
# ============================================================
# CELL 4: Inspect Turn 1 Response
# ============================================================
# The agent correctly acknowledged Seán's name and colour.
# But this response object is ephemeral — nothing is saved.

from pprint import pprint

pprint(response)

{'messages': [HumanMessage(content='Hello my name is Yogi Reddy and my favourite colour is Blue', additional_kwargs={}, response_metadata={}, id='803b67e6-8b2d-4c29-89bf-ef5d40344e33'),
              AIMessage(content="Hello Yogi Reddy! It's nice to meet you.\n\nBlue is a wonderful choice for a favorite color!", additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019f363a-d80f-7681-b61b-044662596fbd-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 13, 'output_tokens': 312, 'total_tokens': 325, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 289}})]}


In [9]:
# ============================================================
# CELL 5: Turn 2 — Ask About the Colour (Memory Fail)
# ============================================================
# This is a NEW .invoke() call — a completely blank slate.
# The agent has NO memory of the previous call.
#
# Watch what happens: the agent will say it doesn't know,
# or make up a generic response — because from its perspective,
# no name or colour was ever mentioned.
#
# This is the core problem that checkpointers solve.

question = HumanMessage(content="What's my favourite colour?")

response = agent.invoke(
    {"messages": [question]}  # Only this one message — no history!
)

pprint(response)

{'messages': [HumanMessage(content="What's my favourite colour?", additional_kwargs={}, response_metadata={}, id='c9f28aba-3736-46a4-9b16-f1fa64630644'),
              AIMessage(content="As an AI, I don't have access to personal information about you, so I can't possibly know your favorite color! You're the only one who knows that.\n\nWhat is it? I'd love to know!", additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019f363b-587b-7d43-9e23-0ce6dd86de70-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 8, 'output_tokens': 632, 'total_tokens': 640, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 583}})]}


---
## Section 2 — With Memory (InMemorySaver)

### InMemorySaver — what it is and its limits

`InMemorySaver` stores conversation history in a **Python dictionary in RAM**. It is the simplest possible checkpointer:

| Property | Detail |
|---|---|
| **Speed** | Instant — no I/O |
| **Persistence** | Lost when the Python process restarts |
| **Use case** | Development, notebooks, demos |
| **Production use** | No — use `SqliteSaver` or `PostgresSaver` instead |

For production apps where memory must survive server restarts, you would swap `InMemorySaver()` for a database-backed checkpointer — but the API (thread_id, config dict) stays exactly the same. That's the beauty of LangGraph's abstraction.

In [13]:
# ============================================================
# CELL 6: Create a Memory-Enabled Agent
# ============================================================
# InMemorySaver stores the conversation history in a Python dict.
# It is keyed by thread_id — each thread gets its own isolated
# message history.
#
# Swapping to a production checkpointer later is one line change:
#   from langgraph.checkpoint.sqlite import SqliteSaver
#   checkpointer = SqliteSaver.from_conn_string("conversations.db")
# Everything else — thread_id, config, .invoke() — stays the same.

from langgraph.checkpoint.memory import InMemorySaver

agent = create_agent(
    "groq:llama-3.1-8b-instant",
    checkpointer=InMemorySaver()  # Attach the memory backend
)

In [14]:
# ============================================================
# CELL 7: Turn 1 — Introduce Yourself (With Memory)
# ============================================================
# Two new things compared to Section 1:
#
# 1. config = {"configurable": {"thread_id": "1"}}
#    This dict is passed as the second argument to .invoke().
#    thread_id="1" labels this conversation. Think of it as a
#    session ID or chat room ID.
#    All calls with the same thread_id share the same history.
#
# 2. config is passed to agent.invoke(..., config)
#    LangGraph reads the thread_id, looks up (or creates) the
#    message history for that thread, and saves the result back
#    when the call finishes.

from langchain.messages import HumanMessage

question = HumanMessage(content="Hello my name is Yogi Reddy and my favourite colour is Blue")

# config tells LangGraph which conversation thread this call belongs to
config = {"configurable": {"thread_id": "1"}}

response = agent.invoke(
    {"messages": [question]},
    config,   # ← thread identifier — this is what enables memory
)

In [15]:
# ============================================================
# CELL 8: Inspect Turn 1 Response
# ============================================================
# Looks the same as before — the agent greeted Seán.
# The difference is invisible: LangGraph has now SAVED this
# entire message history under thread_id="1" in the InMemorySaver.
# The next call with the same thread_id will load it.

pprint(response)

{'messages': [HumanMessage(content='Hello my name is Yogi Reddy and my favourite colour is Blue', additional_kwargs={}, response_metadata={}, id='69ae1810-fa01-4b9d-873d-ef65c619c228'),
              AIMessage(content="Nice to meet you, Yogi Reddy. It's great to know that your favorite color is blue. Blue is a wonderful and calming color, often associated with trust, loyalty, and creativity. Do you have a particular shade of blue that you prefer, or is it just blue in general?", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 61, 'prompt_tokens': 49, 'total_tokens': 110, 'completion_time': 0.087593502, 'completion_tokens_details': None, 'prompt_time': 0.007743083, 'prompt_tokens_details': None, 'queue_time': 0.055178767, 'total_time': 0.095336585}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019f363e-a139-7d10-889e

In [17]:
# ============================================================
# CELL 9: Turn 2 — Ask About the Colour (Memory Works!)
# ============================================================
# Same config (thread_id="1") → LangGraph loads the history from
# the checkpointer and prepends it to this call's messages.
#
# From the model's perspective, the full conversation looks like:
#   [HumanMessage: "Hello my name is Seán..."]   ← from Turn 1
#   [AIMessage:    "Nice to meet you, Seán!"]     ← from Turn 1
#   [HumanMessage: "What's my favourite colour?"] ← current Turn 2
#
# We only pass ONE message in this call, but the agent sees THREE
# because the checkpointer automatically appended the history.
#
# Result: the agent correctly answers "green" — it remembers!

question = HumanMessage(content="What's my favourite colour?")
config = {"configurable": {"thread_id": "1"}}
response = agent.invoke(
    {"messages": [question]},
    config,   # Same thread_id → same memory
)

pprint(response)

{'messages': [HumanMessage(content='Hello my name is Yogi Reddy and my favourite colour is Blue', additional_kwargs={}, response_metadata={}, id='69ae1810-fa01-4b9d-873d-ef65c619c228'),
              AIMessage(content="Nice to meet you, Yogi Reddy. It's great to know that your favorite color is blue. Blue is a wonderful and calming color, often associated with trust, loyalty, and creativity. Do you have a particular shade of blue that you prefer, or is it just blue in general?", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 61, 'prompt_tokens': 49, 'total_tokens': 110, 'completion_time': 0.087593502, 'completion_tokens_details': None, 'prompt_time': 0.007743083, 'prompt_tokens_details': None, 'queue_time': 0.055178767, 'total_time': 0.095336585}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019f363e-a139-7d10-889e

---
## Summary

| Concept | Key takeaway |
|---|---|
| **Stateless default** | Every `.invoke()` without a checkpointer starts a blank conversation |
| **Checkpointer** | A persistence backend that saves/loads conversation history between calls |
| **`InMemorySaver`** | Stores history in RAM — great for dev/notebooks, lost on restart |
| **`thread_id`** | A string key that identifies one conversation — same ID = shared memory |
| **`config` dict** | Passed as second arg to `.invoke()` — tells LangGraph which thread to load/save |
| **Production** | Swap `InMemorySaver` for `SqliteSaver` or `PostgresSaver` — same API, persists across restarts |

### The memory flow (memorise this)

```
agent.invoke({"messages": [new_msg]}, {"configurable": {"thread_id": "X"}})
                                                              │
                          ┌───────────────────────────────────┘
                          ▼
              LangGraph loads history for thread "X"
              prepends it to new_msg
              runs the agent with the full history
              saves the updated history back to thread "X"
```

### Next steps
- **1.4** — Multimodal messages: sending images and audio to agents
- **1.5** — Personal Chef: combining tools + memory in a complete app